# 04 決算データ収集 (Earnings / Financials, J-Quants)

| 項目 | 内容 |
|------|------|
| 目的 | J-Quants APIで JP 株式の四半期決算データを収集し、Earnings Surprise (PEAD) 指標を算出 |
| 出力 | `data/earnings_financials.parquet`, `data/earnings_events.parquet`, `data/ohlcv_with_earnings.parquet` |
| 注意 | `JQUANTS_API_KEY` が未設定/ダミーの場合は空データとなり後続処理は安全にスキップされる |

### 実地検証済みの仕様 (2026-07, 詳細は `funcs/event_collector.py` docstring)
- 財務サマリー: `GET /v2/fins/summary?code=XXXX` (実データ確認済み: 売上・営業利益・純利益・実績EPS・予想EPS)
- `/v2/fins/announcement` (決算予定カレンダー) は**存在しない**。決算発表日は `fins/summary` の
  開示日 (DiscDate) で代替するため、**過去の開示のみ取得可能・将来の予定日は不明**
- Earnings Surprise は「当該開示の実績EPS」と「直前開示時点の予想EPS」の乖離率として算出
  (J-Quantsが実績/予想を直接提供するため、旧版のYoY成長率乖離という代理指標は不要になった)

---
## 0. 環境セットアップ

In [ ]:
import sys
import warnings
from pathlib import Path
from datetime import datetime, timedelta

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import yaml
from tqdm import tqdm

try:
    import japanize_matplotlib
except ImportError:
    print('japanize_matplotlib 未インストール。日本語フォントが文字化けする可能性あり')

sys.path.append(str(Path('../../').resolve()))
from funcs import event_collector, universe_collector, feature_engineering as fe

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.4f}'.format)
%matplotlib inline

print(f'pandas  : {pd.__version__}')
print('Setup OK')

---
## 1. 設定読み込み・銘柄メタ情報 (J-Quants上場銘柄マスタ)

In [ ]:
# ---- 設定ファイル読み込み ----
CFG_PATH = Path('../../configs/stock_config.yaml')
with open(CFG_PATH, encoding='utf-8') as f:
    cfg = yaml.safe_load(f)

DATA_DIR      = Path(cfg['paths']['data'])
PROCESSED_DIR = Path(cfg['paths']['processed_data'])
FIGURES_DIR   = Path(cfg['paths']['figures'])
DATA_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

DATA_START     = cfg['equity']['data_start']
EQUITY_SYMBOLS = cfg['equity']['symbols']
RANDOM_SEED    = cfg.get('random_seed', 42)

print(f'Data start : {DATA_START}')
print(f'Equity     : {len(EQUITY_SYMBOLS)} 銘柄')
print(f'Data dir   : {DATA_DIR.resolve()}')

In [ ]:
# ---- 銘柄名・セクターマッピング (J-Quants上場銘柄マスタから取得、手動管理を廃止) ----
df_universe_meta = universe_collector.fetch_listed_instruments()

if not df_universe_meta.empty:
    df_meta = df_universe_meta[df_universe_meta['symbol'].isin(EQUITY_SYMBOLS)].copy()
    df_meta = df_meta.rename(columns={'sector33_name': 'sector'})
    # config にあってマスタにヒットしなかった銘柄は Unknown で補完
    missing = set(EQUITY_SYMBOLS) - set(df_meta['symbol'])
    if missing:
        df_meta = pd.concat([
            df_meta,
            pd.DataFrame([{'symbol': s, 'name': s, 'market_segment': 'Unknown',
                            'sector33_code': None, 'sector': 'Unknown'} for s in missing]),
        ], ignore_index=True)
else:
    print('JQUANTS_API_KEY が未設定/ダミーのため銘柄マスタが空です。symbol をそのまま name として使用します')
    df_meta = pd.DataFrame([{'symbol': s, 'name': s, 'market_segment': 'Unknown', 'sector': 'Unknown'}
                             for s in EQUITY_SYMBOLS])

SYMBOL_INFO = df_meta.set_index('symbol')[['name', 'sector']].to_dict(orient='index')

print(f'銘柄メタ: {len(df_meta)} 件')
if 'sector' in df_meta.columns:
    print(df_meta['sector'].value_counts())
df_meta.head()

---
## 2. 財務データ・決算イベント収集 (J-Quants)

`event_collector.build_earnings_events()` が財務サマリー取得と実績/予想EPSの突合を内部で
行うため、旧版で個別に実装していた「四半期財務収集」「決算発表日抽出」「Surpriseの手動算出」の
3ステップが1回の呼び出しに集約される。銘柄ごとに1〜数APIコールのため無料プランのレート制限
(約5req/分)を踏まえ、銘柄数が多い場合は数分かかることがある。

In [ ]:
# ---- 財務サマリー (詳細: 売上・営業利益・純利益・EPS実績/予想) を全銘柄収集 ----
financials_raw = {}
errors = {}

for sym in tqdm(EQUITY_SYMBOLS, desc='財務データ収集'):
    df_fin = event_collector.fetch_financial_summary(sym)
    financials_raw[sym] = df_fin
    if df_fin.empty:
        errors[sym] = '空データ (APIキー未設定または開示なし)'

print(f'\n収集完了: {len(EQUITY_SYMBOLS)} 銘柄')
if errors:
    print(f'空データ: {len(errors)} 銘柄')
else:
    print('全銘柄でデータ取得成功')

# ---- 決算イベント (実績/予想EPS・Surprise) を全銘柄まとめて収集 ----
df_earnings_events = event_collector.build_earnings_events(EQUITY_SYMBOLS)
print(f'\n決算イベント: {len(df_earnings_events):,} 件')
if not df_earnings_events.empty:
    display(df_earnings_events.head(10))

---
## 3. 財務データの整形 (YoY成長率算出)

In [ ]:
# ---- financials_raw (symbol -> DataFrame) をフラット化し、YoY成長率(4期前比)を付与 ----
records = []
for sym, df_fin in financials_raw.items():
    if df_fin.empty:
        continue
    rec = df_fin.sort_values('disc_date').reset_index(drop=True).copy()
    for col in ['sales', 'op', 'np']:
        rec[f'{col}_yoy'] = rec[col].pct_change(4)
    records.append(rec)

if records:
    df_financials = pd.concat(records, ignore_index=True)
    df_financials = df_financials.merge(df_meta[['symbol', 'name', 'sector']], on='symbol', how='left')
    df_financials = df_financials.sort_values(['symbol', 'disc_date']).reset_index(drop=True)
    print(f'四半期財務 DataFrame: {df_financials.shape}')
    print(f'銘柄数: {df_financials["symbol"].nunique()}')
    print(f'期間  : {df_financials["disc_date"].min().date()} ~ {df_financials["disc_date"].max().date()}')
    df_financials.head()
else:
    print('警告: 財務データを取得できた銘柄がありません')
    df_financials = pd.DataFrame()

In [ ]:
---
## 4. Earnings Surprise のクロスセクション正規化

`surprise_pct` は既に `build_earnings_events()` が実績EPS vs 直前予想EPSの乖離率として算出済み。
銘柄間でのスケール差を吸収するため、開示日ごとにクロスセクションZスコア化する
(PEADのシグナルとして利用する際は生値よりZスコアの方が扱いやすい)。

In [ ]:
if not df_earnings_events.empty:
    # 開示日ごとにクロスセクションZスコア化 (同日開示が少数の場合はNaNになりうる)
    df_earnings_events['surprise_z'] = df_earnings_events.groupby('announcement_date')['surprise_pct'].transform(
        lambda x: (x - x.mean()) / (x.std() + 1e-8) if x.notna().sum() >= 2 else np.nan
    )

    n_valid = df_earnings_events['surprise_pct'].notna().sum()
    print(f'surprise_pct 有効件数: {n_valid} / {len(df_earnings_events)}')
    print(f'surprise_z   有効件数: {df_earnings_events["surprise_z"].notna().sum()} / {len(df_earnings_events)}')

    display_cols = ['symbol', 'announcement_date', 'fiscal_period', 'actual_eps', 'forecast_eps',
                     'surprise_pct', 'surprise_z']
    print('\nSurprise 指標サンプル (上位|下位 surprise_pct):')
    valid = df_earnings_events.dropna(subset=['surprise_pct'])
    if not valid.empty:
        display(pd.concat([valid.nlargest(5, 'surprise_pct'), valid.nsmallest(5, 'surprise_pct')])[display_cols])
else:
    print('警告: df_earnings_events が空のため Surprise 正規化をスキップ')

---
## 5. Earnings EDA (探索的データ分析)

In [ ]:
# ---- 6-1. 売上高 YoY 成長率のヒストグラム ----
if not df_financials.empty and 'sales_yoy' in df_financials.columns:
    fig, ax = plt.subplots(figsize=(10, 5))
    data_yoy = df_financials['sales_yoy'].dropna()
    # 外れ値を除外して表示 (±200%)
    data_yoy_clipped = data_yoy.clip(-2.0, 2.0)
    ax.hist(data_yoy_clipped, bins=40, color='steelblue', alpha=0.7, edgecolor='white')
    ax.axvline(0, color='red', linestyle='--', linewidth=1.5, label='YoY=0%')
    ax.axvline(data_yoy.median(), color='orange', linestyle='--', linewidth=1.5,
               label=f'中央値: {data_yoy.median():.1%}')
    ax.set_xlabel('売上高 YoY 成長率')
    ax.set_ylabel('頻度')
    ax.set_title('売上高 YoY 成長率の分布 (全銘柄・全期間、±200%クリップ)')
    ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x:.0%}'))
    ax.legend()
    ax.grid(alpha=0.3)
    plt.tight_layout()
    save_path = FIGURES_DIR / '04_revenue_yoy_hist.png'
    plt.savefig(save_path, dpi=120, bbox_inches='tight')
    plt.show()
    print(f'保存: {save_path}')
else:
    print('sales_yoy データなし: スキップ')

In [ ]:
# ---- 6-2. セクター別 売上高 YoY 成長率 箱ひげ図 ----
if not df_financials.empty and 'sales_yoy' in df_financials.columns and 'sector' in df_financials.columns:
    sector_data = {}
    for sec, grp in df_financials.groupby('sector'):
        vals = grp['sales_yoy'].dropna().clip(-1.5, 1.5).values
        if len(vals) >= 3:
            sector_data[sec] = vals

    if sector_data:
        sectors_sorted = sorted(sector_data.keys(),
                                key=lambda s: np.median(sector_data[s]),
                                reverse=True)
        fig, ax = plt.subplots(figsize=(12, 6))
        bp = ax.boxplot(
            [sector_data[s] for s in sectors_sorted],
            labels=sectors_sorted,
            patch_artist=True,
            medianprops=dict(color='red', linewidth=2),
        )
        colors = plt.cm.Set3(np.linspace(0, 1, len(sectors_sorted)))
        for patch, color in zip(bp['boxes'], colors):
            patch.set_facecolor(color)
        ax.axhline(0, color='gray', linestyle='--', linewidth=1)
        ax.set_xticklabels(sectors_sorted, rotation=30, ha='right')
        ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x:.0%}'))
        ax.set_ylabel('売上高 YoY 成長率')
        ax.set_title('セクター別 売上高 YoY 成長率 (J-Quants 33業種区分)')
        ax.grid(axis='y', alpha=0.3)
        plt.tight_layout()
        save_path = FIGURES_DIR / '04_sector_revenue_yoy_box.png'
        plt.savefig(save_path, dpi=120, bbox_inches='tight')
        plt.show()
        print(f'保存: {save_path}')
    else:
        print('セクター別データ不足: スキップ')
else:
    print('データなし: スキップ')

In [ ]:
# ---- 5-3. 上位5銘柄の売上高トレンド (折れ線グラフ) ----
if not df_financials.empty and 'sales' in df_financials.columns:
    # 最新期末の売上高が大きい銘柄 Top 5 を選択
    latest_rev = (
        df_financials
        .dropna(subset=['sales'])
        .sort_values('disc_date')
        .groupby('symbol')
        .last()
        ['sales']
        .nlargest(5)
    )
    top5 = latest_rev.index.tolist()

    fig, ax = plt.subplots(figsize=(14, 6))
    colors = plt.cm.tab10(np.linspace(0, 0.5, len(top5)))
    for sym, color in zip(top5, colors):
        df_s = df_financials[df_financials['symbol'] == sym].dropna(subset=['sales'])
        name = SYMBOL_INFO.get(sym, {}).get('name', sym)
        ax.plot(df_s['disc_date'], df_s['sales'] / 1e9,
                marker='o', markersize=4, linewidth=1.5,
                label=f'{sym} ({name})', color=color)

    ax.set_xlabel('開示日')
    ax.set_ylabel('売上高 (十億円)')
    ax.set_title('売上高トレンド (Top 5 銘柄, 四半期)')
    ax.legend(fontsize=9)
    ax.grid(alpha=0.3)
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
    plt.xticks(rotation=30)
    plt.tight_layout()
    save_path = FIGURES_DIR / '04_top5_revenue_trend.png'
    plt.savefig(save_path, dpi=120, bbox_inches='tight')
    plt.show()
    print(f'Top5: {top5}')
    print(f'保存: {save_path}')
else:
    print('sales データなし: スキップ')

In [ ]:
---
## 6. OHLCVデータとのマージ

`funcs.feature_engineering.add_event_features()` (Application backend と共通のロジック) を
再利用し、決算近接度・直近サプライズを OHLCV に付与する。リーク防止のため
`earnings_surprise_pct` は発表翌営業日以降のみ有効になる (関数内部でシフト済み)。

In [ ]:
# ---- 6-1. OHLCV 読み込み ----
ohlcv_path = DATA_DIR / 'equity_jp_ohlcv.parquet'

if ohlcv_path.exists():
    df_ohlcv = pd.read_parquet(ohlcv_path)
    df_ohlcv['date'] = pd.to_datetime(df_ohlcv['date']).dt.tz_localize(None)
    print(f'OHLCV 読み込み: {df_ohlcv.shape}')
    print(f'期間: {df_ohlcv["date"].min().date()} ~ {df_ohlcv["date"].max().date()}')
    df_ohlcv.head()
else:
    print(f'警告: {ohlcv_path} が見つかりません。先に 03_equity_data_collection.ipynb を実行してください。')
    df_ohlcv = pd.DataFrame()

In [ ]:
# ---- 6-2. feature_engineering.add_event_features で決算特徴量を付与 ----
if not df_ohlcv.empty and not df_earnings_events.empty:
    df_ohlcv_merged = fe.add_event_features(
        df_ohlcv.copy(),
        df_earnings_events[['symbol', 'announcement_date', 'surprise_pct']],
    )
    n_upcoming = df_ohlcv_merged['days_to_next_earnings'].notna().sum()
    n_surprise = df_ohlcv_merged['earnings_surprise_pct'].notna().sum()
    print(f'\ndays_to_next_earnings 付与行数: {n_upcoming:,}')
    print(f'earnings_surprise_pct 付与行数: {n_surprise:,}')
    print(f'出力 DataFrame: {df_ohlcv_merged.shape}')
    display(df_ohlcv_merged.dropna(subset=['earnings_surprise_pct']).head(10)[
        ['symbol', 'date', 'close', 'days_to_next_earnings', 'days_since_last_earnings', 'earnings_surprise_pct']
    ])
else:
    print('OHLCVまたは決算イベントデータがないためマージをスキップ')
    df_ohlcv_merged = pd.DataFrame()

---
## 7. データ保存

In [ ]:
if not df_financials.empty:
    out_path = DATA_DIR / 'earnings_financials.parquet'
    df_financials.to_parquet(out_path, index=False)
    print(f'保存: {out_path}  ({len(df_financials):,} 行)')
else:
    print('警告: df_financials が空のため保存をスキップ')

if not df_earnings_events.empty:
    out_path = DATA_DIR / 'earnings_events.parquet'
    df_earnings_events.to_parquet(out_path, index=False)
    print(f'保存: {out_path}  ({len(df_earnings_events):,} 行)')
else:
    print('警告: df_earnings_events が空のため保存をスキップ')

if not df_ohlcv_merged.empty:
    out_path = DATA_DIR / 'ohlcv_with_earnings.parquet'
    df_ohlcv_merged.to_parquet(out_path, index=False)
    print(f'保存: {out_path}  ({len(df_ohlcv_merged):,} 行)')
else:
    print('警告: ohlcv_with_earnings が空のため保存をスキップ')

print('\n全保存完了')

---
## 8. データ品質サマリー

In [ ]:
# ---- 8-1. 銘柄別カバレッジ・欠損率レポート ----
print('=' * 60)
print('銘柄別 財務データ カバレッジ')
print('=' * 60)
coverage_rows = []
for sym in EQUITY_SYMBOLS:
    df_fin = financials_raw.get(sym)
    n = len(df_fin) if df_fin is not None and not df_fin.empty else 0
    coverage_rows.append({'symbol': sym, 'name': SYMBOL_INFO.get(sym, {}).get('name', sym), 'n_disclosures': n})
df_coverage = pd.DataFrame(coverage_rows)
print(df_coverage.to_string(index=False))

print()
print('=' * 60)
print('四半期財務 DataFrame 欠損率')
print('=' * 60)
if not df_financials.empty:
    numeric_cols = df_financials.select_dtypes(include='number').columns.tolist()
    missing_rate = (df_financials[numeric_cols].isnull().mean() * 100).sort_values(ascending=False)
    print(missing_rate.map(lambda x: f'{x:.1f}%').to_string())
else:
    print('データなし')

print()
print('=' * 60)
print('統計サマリー')
print('=' * 60)
print(f'収集成功銘柄数  : {df_coverage[df_coverage["n_disclosures"] > 0].shape[0]} / {len(EQUITY_SYMBOLS)}')
if not df_financials.empty:
    print(f'総開示レコード  : {len(df_financials):,}')
    print(f'平均開示数/銘柄 : {df_financials.groupby("symbol").size().mean():.1f}')
    print(f'期間: {df_financials["disc_date"].min().date()} ~ {df_financials["disc_date"].max().date()}')
if not df_earnings_events.empty:
    n_surprise_valid = df_earnings_events['surprise_pct'].notna().sum()
    print(f'決算イベント     : {len(df_earnings_events):,} 件 (surprise_pct算出済み: {n_surprise_valid:,} 件)')
if not df_ohlcv_merged.empty:
    n_surprise_days = df_ohlcv_merged['earnings_surprise_pct'].notna().sum()
    print(f'earnings_surprise_pct 付与済み: {n_surprise_days:,} 日')

In [ ]:
# ---- 8-2. カバレッジ可視化 ----
if not df_coverage.empty and df_coverage['n_disclosures'].sum() > 0:
    df_cov_sorted = df_coverage.sort_values('n_disclosures', ascending=True)
    fig, ax = plt.subplots(figsize=(12, 7))
    y_pos = range(len(df_cov_sorted))
    ax.barh(y_pos, df_cov_sorted['n_disclosures'], color='steelblue', alpha=0.8)
    ax.set_yticks(list(y_pos))
    ax.set_yticklabels(
        [f"{r['symbol']} ({r['name']})" for _, r in df_cov_sorted.iterrows()],
        fontsize=9
    )
    ax.set_xlabel('開示件数 (財務サマリー)')
    ax.set_title('銘柄別 決算開示データ カバレッジ')
    ax.grid(axis='x', alpha=0.3)
    plt.tight_layout()
    save_path = FIGURES_DIR / '04_coverage_summary.png'
    plt.savefig(save_path, dpi=120, bbox_inches='tight')
    plt.show()
    print(f'保存: {save_path}')
else:
    print('カバレッジデータなし: スキップ')

print('\n次のステップ: 42_equity_lgbm_walkforward.ipynb で決算特徴量(days_to_next_earnings, earnings_surprise_pct)を組み込む')